In [ ]:
!pip install openai langchain pypdf2 faiss-cpu python-dotenv


In [ ]:
# Cell 2: Importaciones y configuración de OpenAI
import os
import json
import pandas as pd
from pathlib import Path

import openai
from langchain.schema import Document
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader, CSVLoader

# Configura tu API key (o usa variables de entorno)
openai.api_key = os.getenv("OPENAI_API_KEY", "TU_API_KEY_AQUI")


In [ ]:
# Cell 4: Paths & load listing of informes
PDF_FOLDER   = "/content/drive/My Drive/Univalle/tesis/Tesis/reports-pdf"  # ajusta según tu ruta
LISTING_PATH = "/content/drive/My Drive/Univalle/tesis/Tesis/resources/listado-informes.xlsx"
listing_df   = pd.read_excel(LISTING_PATH)


In [ ]:
# Cell 5: Initialize embedding model
# Usaremos las embeddings de OpenAI (text-embedding-ada-002)
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")


In [ ]:
# Cell 6: Definir función para seleccionar los informes más relevantes con GPT-4.1
def get_relevant_reports_openai(question: str, k: int = 3) -> list[str]:
    """
    Dada una pregunta y el DataFrame listing_df con 'ident', 'title', 'description',
    devuelve los k IDs de informe más relevantes usando GPT-4.1 y JSON.
    """
    informes = listing_df[['ident', 'title', 'description']].to_dict(orient='records')
    prompt = f"""
Tienes esta lista de informes (identificador, título y descripción):
{json.dumps(informes, ensure_ascii=False, indent=2)}

Dada la pregunta:
"{question}"

Devuélveme **únicamente** los {k} identificadores de los informes más relevantes,
en formato JSON así:

{{ "top_reports": ["ID1", "ID2", ..., "ID{k}"] }}
"""
    resp = openai.ChatCompletion.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": "Eres un selector de informes. Contesta solo con el JSON solicitado."},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.0
    )
    texto = resp.choices[0].message.content
    salida = json.loads(texto)
    return salida.get("top_reports", [])


In [ ]:
# Cell 7: Definir función para extraer contextos de un informe dado
def retrieve_report_context(ident: str, question: str, k_chunks: int = 5) -> list[str]:
    """
    Carga el PDF o CSV del informe, divide en chunks, construye un FAISS
    vectorstore in-memory y recupera los k_chunks más similares a la pregunta.
    """
    pdf_path = os.path.join(PDF_FOLDER, f"{ident}.pdf")
    csv_path = os.path.join(PDF_FOLDER, f"{ident}.csv")
    if os.path.exists(pdf_path):
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
    elif os.path.exists(csv_path):
        loader = CSVLoader(csv_path)
        docs = loader.load()
    else:
        return []
    # dividir en trozos
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(docs)
    # índice FAISS
    vectordb = FAISS.from_documents(chunks, embeddings)
    results = vectordb.similarity_search(question, k=k_chunks)
    return [doc.page_content for doc in results]


In [ ]:
# Cell 8: Agrupar contexto de los top-k informes
def retrieve_contexts(question: str, top_reports: int = 3, top_chunks: int = 5) -> list[str]:
    """
    Usa get_relevant_reports_openai para obtener top_reports IDs,
    luego extrae hasta top_chunks contextos de cada uno.
    """
    top_ids = get_relevant_reports_openai(question, k=top_reports)
    all_ctx = []
    for ident in top_ids:
        all_ctx.extend(retrieve_report_context(ident, question, k_chunks=top_chunks))
    return all_ctx


In [ ]:
# Cell 9: Función principal para responder la pregunta
def answer_question(question: str, top_reports: int = 3, top_chunks: int = 5, max_tokens: int = 512) -> str:
    """
    Pipeline completo: selecciona informes, recupera contexto y genera
    la respuesta usando GPT-4.1.
    """
    contexts = retrieve_contexts(question, top_reports=top_reports, top_chunks=top_chunks)
    context_text = "\n\n---\n\n".join(contexts)
    prompt = f"""
Eres un experto en el conflicto armado colombiano. Basándote en el siguiente contexto,
responde de forma clara y precisa a la pregunta.

Pregunta:
{question}

Contexto relevante:
{context_text}

Por favor, brinda tu respuesta:
"""
    resp = openai.ChatCompletion.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": "Eres un historiador experto y respondes con detalle y precisión."},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.0,
        max_tokens=max_tokens
    )
    return resp.choices[0].message.content.strip()


In [ ]:
pregunta = (
    "¿Cuáles son los hallazgos principales del informe "
    "058-CI-00235 Fuego en el remanso de paz: aproximaciones a la memoria colectiva "
    "de las víctimas del conflicto armado en el municipio de Titiribí?"
)
respuesta = answer_question(pregunta, top_reports=3, top_chunks=5, max_tokens=512)
print("Respuesta:\n", respuesta)